In [8]:
#!/usr/bin/env python3
# =========================================================
# 🏛️  HYBRID LEGAL SUMMARIZATION — VERSION 2 (FULL DATASET)
#
# ✅ CHANGE: Role annotations for ALL documents (train+test)
#    No sample cap — processes the full 7030-document dataset
#
# KEY FEATURES:
#   • Checkpoint saving every CHECKPOINT_EVERY documents
#     → Safe to Ctrl+C and resume — won't re-annotate done docs
#   • Batch-level progress bar with ETA
#   • Annotates BOTH train split and test split fully
#   • All downstream steps (scoring, weights, generation,
#     ensemble, evaluation) unchanged
# =========================================================

import os
import sys
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from collections import Counter
from tqdm import tqdm

import nltk
try:
    nltk.data.find("tokenizers/punkt")
except LookupError:
    nltk.download("punkt", quiet=True)

from nltk.tokenize import sent_tokenize
from sklearn.metrics.pairwise import cosine_similarity
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSeq2SeqLM,
    LEDForConditionalGeneration,
    PegasusForConditionalGeneration,
)
import evaluate

# ─────────────────────────────────────────────────────────
# DEVICE
# ─────────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🖥️  Device: {device}")

# ─────────────────────────────────────────────────────────
# PATHS  — all files in same folder
# ─────────────────────────────────────────────────────────
BASE_DIR   = "."                                      # ← folder where this script lives
OUTPUT_DIR = os.path.join(BASE_DIR, "results_cross_attention_ensemble")
os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_JSON_PATH = os.path.join(BASE_DIR, "train.json")
TEST_JSON_PATH  = os.path.join(BASE_DIR, "test.json")

MODEL_PATHS = {
    "BART":       os.path.join(BASE_DIR, "BART"),
    "LED":        os.path.join(BASE_DIR, "LED"),
    "PEGASUS":    os.path.join(BASE_DIR, "PEGASUS"),
    "ROLE_MODEL": os.path.join(BASE_DIR, "best_RR_model"),
}

ROLE_ANNOTATION_FILE   = "sentence_role_annotations_8label.json"
ROLE_CONTRIBUTION_FILE = "role_contribution_scores_8label.json"
ROLE_WEIGHTS_FILE      = "normalized_role_weights_8label.json"

# ─────────────────────────────────────────────────────────
# ✅  FULL DATASET — NO SAMPLE CAP
# ─────────────────────────────────────────────────────────
MAX_TRAIN_SAMPLES  = None   # None = use ALL training documents

# Checkpoint: save partial annotations every N docs
# Allows safe resume if the job is interrupted
CHECKPOINT_EVERY   = 100    # save progress every 100 docs

# ─────────────────────────────────────────────────────────
# MODEL CONFIGURATION
# ─────────────────────────────────────────────────────────
MODEL_CONFIG = {
    "BART":    {"max_input": 1024, "max_output": 256, "num_beams": 4},
    "LED":     {"max_input": 4096, "max_output": 512, "num_beams": 4},
    "PEGASUS": {"max_input": 1024, "max_output": 256, "num_beams": 4},
}

HYBRID_ALPHA     = 0.6
HYBRID_BETA      = 0.4
ENSEMBLE_WEIGHTS = {"BART": 0.25, "LED": 0.50, "PEGASUS": 0.25}

RAG_ANCHOR_ROLES    = ["ISSUE", "REASONING"]
RAG_ANCHOR_FRACTION = 0.20
RAG_ROLE_BUDGET     = 50
RAG_MODELS          = {"BART", "PEGASUS"}

COMPRESSION = {"BART": 0.12, "PEGASUS": 0.12, "LED": 0.40}
MIN_SENTS   = {"BART": 30,   "PEGASUS": 30,   "LED": 100}
MAX_SENTS   = {"BART": 150,  "PEGASUS": 150,  "LED": 400}

# ─────────────────────────────────────────────────────────
# LABEL SYSTEM  (13 HSLN → 8 strategic labels)
# ─────────────────────────────────────────────────────────
HSLN_LABELS = [
    "PREAMBLE", "FAC", "RLC", "ISSUE", "ARG_PETITIONER", "ARG_RESPONDENT",
    "ANALYSIS", "STA", "PRE_RELIED", "PRE_NOT_RELIED", "RATIO", "RPC", "NONE",
]
NUM_HSLN_LABELS = 13

LABELS_8 = [
    "PREAMBLE", "FACTS", "ISSUE", "ARGUMENTS",
    "REASONING", "PRECEDENT_RELIED", "PRECEDENT_NOT_RELIED", "PROCEDURAL",
]

LABEL_MAP_13_TO_8 = {
    "PREAMBLE":       "PREAMBLE",
    "ISSUE":          "ISSUE",
    "FAC":            "FACTS",
    "PRE_RELIED":     "PRECEDENT_RELIED",
    "PRE_NOT_RELIED": "PRECEDENT_NOT_RELIED",
    "ARG_PETITIONER": "ARGUMENTS",
    "ARG_RESPONDENT": "ARGUMENTS",
    "ANALYSIS":       "REASONING",
    "RATIO":          "REASONING",
    "RPC":            "REASONING",
    "STA":            "PROCEDURAL",
    "RLC":            "PROCEDURAL",
    "NONE":           "PROCEDURAL",
}

NARRATIVE_ORDER = {
    "PREAMBLE": 0, "FACTS": 1, "ISSUE": 2, "ARGUMENTS": 3,
    "PRECEDENT_RELIED": 4, "PRECEDENT_NOT_RELIED": 5,
    "REASONING": 6, "PROCEDURAL": 7,
}

def map_13_to_8(labels_13):
    return [LABEL_MAP_13_TO_8.get(l, "PROCEDURAL") for l in labels_13]

# ─────────────────────────────────────────────────────────
# HSLN ROLE CLASSIFIER  (13 labels)
# ─────────────────────────────────────────────────────────
class PrototypeAttention(nn.Module):
    def __init__(self, dim, heads=8, dropout=0.1):
        super().__init__()
        self.h  = heads
        self.dh = dim // heads
        self.q  = nn.Linear(dim, dim, bias=False)
        self.k  = nn.Linear(dim, dim, bias=False)
        self.v  = nn.Linear(dim, dim, bias=False)
        self.o  = nn.Linear(dim, dim, bias=False)
        self.ln = nn.LayerNorm(dim)
        self.dp = nn.Dropout(dropout)

    def forward(self, x, p):
        B, T, D = x.shape
        C = p.size(0)
        Q = self.q(x).view(B, T, self.h, self.dh)
        K = self.k(p).view(C, self.h, self.dh)
        V = self.v(p).view(C, self.h, self.dh)
        Q = F.normalize(Q, dim=-1)
        K = F.normalize(K, dim=-1)
        outs = []
        for h in range(self.h):
            a = (Q[:, :, h] @ K[:, h].T).softmax(-1)
            a = self.dp(a)
            outs.append(a @ V[:, h])
        out = torch.cat(outs, -1)
        return self.ln(x + self.dp(self.o(out)))

class RPL(nn.Module):
    def __init__(self, dim, num_labels):
        super().__init__()
        self.proto = nn.Parameter(torch.randn(num_labels, dim))
    def forward(self, h):
        return F.normalize(h, dim=-1) @ F.normalize(self.proto, dim=-1).T

class RTM(nn.Module):
    def __init__(self, num_labels, lam):
        super().__init__()
        self.A   = nn.Parameter(torch.zeros(num_labels, num_labels))
        self.lam = lam
    def forward(self, logits):
        lp = logits.log_softmax(-1)
        for t in range(1, logits.size(1)):
            tr = torch.logsumexp(lp[:, t-1].unsqueeze(1) + self.A.log_softmax(-1), -1)
            logits[:, t] += self.lam * tr
        return logits

class HSLNModel(nn.Module):
    def __init__(self, embedding_dim=768, lstm_hidden=384, num_labels=13,
                 dropout=0.3, rtm_lambda=0.05):
        super().__init__()
        self.att   = PrototypeAttention(embedding_dim, dropout=dropout)
        self.lstm  = nn.LSTM(embedding_dim, lstm_hidden, 2,
                             bidirectional=True, batch_first=True, dropout=dropout)
        self.cls   = nn.Linear(lstm_hidden * 2, num_labels)
        self.rpl   = RPL(lstm_hidden * 2, num_labels)
        self.alpha = nn.Parameter(torch.tensor(2.0))
        self.rtm   = RTM(num_labels, rtm_lambda)
        self.register_buffer("prototypes", torch.randn(num_labels, embedding_dim))

    def forward(self, embeddings, lengths=None):
        x = self.att(embeddings, self.prototypes)
        if lengths is not None:
            pck = nn.utils.rnn.pack_padded_sequence(
                x, lengths.cpu(), batch_first=True, enforce_sorted=False)
            o, _ = self.lstm(pck)
            o, _ = nn.utils.rnn.pad_packed_sequence(o, batch_first=True)
        else:
            o, _ = self.lstm(x)
        l1 = self.cls(o)
        l2 = self.rpl(o)
        a  = torch.sigmoid(self.alpha)
        return self.rtm(a * l1 + (1 - a) * l2)

    def predict(self, embeddings, lengths=None):
        return torch.argmax(self.forward(embeddings, lengths), dim=-1)

# ─────────────────────────────────────────────────────────
# CROSS-SENTENCE ATTENTION SCORER
# ─────────────────────────────────────────────────────────
class CrossSentenceAttentionScorer(nn.Module):
    def __init__(self, embed_dim=768, d_model=128, num_heads=4,
                 num_layers=2, dropout=0.1, max_sentences=1024):
        super().__init__()
        self.input_proj = nn.Linear(embed_dim, d_model)
        self.pos_enc    = nn.Embedding(max_sentences, d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=num_heads,
            dim_feedforward=d_model * 4, dropout=dropout,
            batch_first=True, norm_first=True,
        )
        self.transformer   = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.salience_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(d_model // 2, 1),
        )

    def forward(self, embeddings):
        N   = embeddings.size(0)
        pos = torch.arange(N, device=embeddings.device)
        x   = self.input_proj(embeddings) + self.pos_enc(pos)
        x   = self.transformer(x.unsqueeze(0)).squeeze(0)
        return torch.sigmoid(self.salience_head(x).squeeze(-1))

# ─────────────────────────────────────────────────────────
# LOAD MODELS
# ─────────────────────────────────────────────────────────
print("\n📚 Loading models...")

print("  [1/5] InLegalBERT...")
legal_tok   = AutoTokenizer.from_pretrained("law-ai/InLegalBERT")
legal_model = AutoModel.from_pretrained("law-ai/InLegalBERT").to(device)
legal_model.eval()

print("  [2/5] Cross-Sentence Attention Scorer...")
ca_scorer = CrossSentenceAttentionScorer().to(device)
ca_scorer.eval()

print("  [3/5] HSLN Role Classifier...")
config_path = os.path.join(MODEL_PATHS["ROLE_MODEL"], "config.json")
if os.path.exists(config_path):
    with open(config_path) as f:
        cfg = json.load(f)
    emb_dim = cfg.get("embedding_dim", 768)
    lstm_hid = cfg.get("lstm_hidden", 384)
    drop     = cfg.get("dropout", 0.3)
    rtm_lam  = cfg.get("rtm_lambda", 0.05)
else:
    emb_dim, lstm_hid, drop, rtm_lam = 768, 384, 0.3, 0.05

role_model = HSLNModel(emb_dim, lstm_hid, NUM_HSLN_LABELS, drop, rtm_lam)
state_dict = torch.load(os.path.join(MODEL_PATHS["ROLE_MODEL"], "pytorch_model.bin"),
                        map_location=device)
state_dict.pop("prototypes", None)
role_model.load_state_dict(state_dict, strict=False)
role_model.prototypes = torch.load(
    os.path.join(MODEL_PATHS["ROLE_MODEL"], "prototypes.pt"), map_location=device)
role_model.to(device).eval()

print("  [4/5] BART + LED + PEGASUS...")
summ_models, summ_toks = {}, {}
summ_toks["BART"]      = AutoTokenizer.from_pretrained(MODEL_PATHS["BART"])
summ_models["BART"]    = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_PATHS["BART"]).to(device).eval()
summ_toks["LED"]       = AutoTokenizer.from_pretrained(MODEL_PATHS["LED"])
summ_models["LED"]     = LEDForConditionalGeneration.from_pretrained(
    MODEL_PATHS["LED"]).to(device).eval()
summ_toks["PEGASUS"]   = AutoTokenizer.from_pretrained(MODEL_PATHS["PEGASUS"])
summ_models["PEGASUS"] = PegasusForConditionalGeneration.from_pretrained(
    MODEL_PATHS["PEGASUS"]).to(device).eval()

print("  [5/5] Evaluation metrics...")
rouge_metric     = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")

print("✅ All models loaded.\n")

# ─────────────────────────────────────────────────────────
# EMBEDDING
# ─────────────────────────────────────────────────────────
@torch.no_grad()
def embed_sentences(texts: list, batch_size: int = 16) -> np.ndarray:
    if not texts:
        return np.zeros((0, 768), dtype=np.float32)
    all_embs = []
    for i in range(0, len(texts), batch_size):
        enc  = legal_tok(texts[i:i+batch_size], padding=True, truncation=True,
                         max_length=512, return_tensors="pt").to(device)
        out  = legal_model(**enc).last_hidden_state
        mask = enc["attention_mask"].unsqueeze(-1).float()
        all_embs.append(((out * mask).sum(1) / mask.sum(1)).cpu().numpy())
    return np.vstack(all_embs)

# ─────────────────────────────────────────────────────────
# CROSS-ATTENTION SALIENCY
# ─────────────────────────────────────────────────────────
@torch.no_grad()
def compute_saliency(sentences: list, embeddings: np.ndarray) -> np.ndarray:
    if not sentences:
        return np.array([], dtype=np.float32)
    raw = ca_scorer(torch.FloatTensor(embeddings).to(device)).cpu().numpy()
    N   = len(sentences)
    pos  = np.arange(N) / max(N - 1, 1)
    bias = 0.8 + 0.2 * (2 * np.abs(pos - 0.5))      # U-shaped
    lpen = np.array([min(1.0, len(s.split()) / 8.0) for s in sentences], dtype=np.float32)
    sal  = raw * bias * lpen
    lo, hi = sal.min(), sal.max()
    return ((sal - lo) / (hi - lo)).astype(np.float32) if hi > lo \
           else np.full(N, 0.5, dtype=np.float32)

# ─────────────────────────────────────────────────────────
# ROLE CLASSIFICATION
# ─────────────────────────────────────────────────────────
@torch.no_grad()
def classify_roles(sentences: list, batch_size: int = 32) -> list:
    if not sentences:
        return []
    preds_13 = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i + batch_size]
        enc   = legal_tok(batch, padding=True, truncation=True,
                          max_length=128, return_tensors="pt").to(device)
        embs  = legal_model(**enc).last_hidden_state.mean(1).unsqueeze(1)
        lens  = torch.ones(len(batch), dtype=torch.long).to(device)
        pred  = role_model.predict(embs, lens)[:, 0].cpu().tolist()
        preds_13.extend(pred)
    return map_13_to_8([HSLN_LABELS[p] for p in preds_13])

# ─────────────────────────────────────────────────────────
# ADAPTIVE TARGETS
# ─────────────────────────────────────────────────────────
def adaptive_targets(doc_len: int) -> dict:
    return {m: max(MIN_SENTS[m], min(MAX_SENTS[m], int(doc_len * r)))
            for m, r in COMPRESSION.items()}

# ─────────────────────────────────────────────────────────
# ✅ STEP 1: FULL-DATASET ROLE ANNOTATION WITH CHECKPOINTING
# ─────────────────────────────────────────────────────────
def annotate_documents_full(data: list, out_path: str,
                             split_name: str = "data") -> list:
    """
    Annotate every sentence in every document with its 8-label role.

    ✅ Processes ALL documents — no sample cap.
    ✅ Checkpoint: saves partial results every CHECKPOINT_EVERY docs.
       If out_path already has N docs, resumes from doc N+1.
    ✅ Progress: tqdm bar with per-doc sentence count.
    """
    checkpoint_path = out_path.replace(".json", "_checkpoint.json")

    # ── Resume from checkpoint if available ──────────────────────
    done_ids  = set()
    completed = []

    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, encoding="utf-8") as f:
            completed = json.load(f)
        done_ids = {ann["doc_id"] for ann in completed}
        print(f"  ♻️  Resuming {split_name}: {len(completed)} docs already done, "
              f"{len(data) - len(completed)} remaining")
    elif os.path.exists(out_path):
        # Final file already exists — load it directly
        with open(out_path, encoding="utf-8") as f:
            completed = json.load(f)
        print(f"  ✅ {split_name} annotation already complete "
              f"({len(completed)} docs). Loading existing file.")
        return completed

    remaining = [item for item in data
                 if item.get("id", data.index(item)) not in done_ids]

    if not remaining:
        print(f"  ✅ All {len(data)} {split_name} docs already annotated.")
        if os.path.exists(out_path):
            with open(out_path, encoding="utf-8") as f:
                return json.load(f)
        # Write final file from checkpoint
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(completed, f, indent=2, ensure_ascii=False)
        return completed

    print(f"\n  📝 Annotating {split_name}: {len(remaining)} docs "
          f"(total dataset = {len(data)})")

    total_sents_processed = sum(
        ann["total_sentences"] for ann in completed)

    with tqdm(total=len(remaining), desc=f"  Annotating {split_name}",
              unit="doc") as pbar:

        for idx, item in enumerate(remaining):
            doc_id = item.get("id", idx)
            sents  = sent_tokenize(item.get("judgment", ""))
            roles  = classify_roles(sents)

            completed.append({
                "doc_id":           doc_id,
                "total_sentences":  len(sents),
                "sentences": [
                    {"index": i, "sentence": s, "role": r}
                    for i, (s, r) in enumerate(zip(sents, roles))
                ],
            })

            total_sents_processed += len(sents)
            pbar.set_postfix({
                "sents_total": total_sents_processed,
                "roles": len(set(roles)),
            })
            pbar.update(1)

            # ── Checkpoint every N documents ──────────────────
            if (idx + 1) % CHECKPOINT_EVERY == 0:
                with open(checkpoint_path, "w", encoding="utf-8") as f:
                    json.dump(completed, f, ensure_ascii=False)
                tqdm.write(f"  💾 Checkpoint saved: {len(completed)} / {len(data)} docs")

    # ── Save final annotation file ────────────────────────────
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(completed, f, indent=2, ensure_ascii=False)

    # Remove checkpoint once final file is written
    if os.path.exists(checkpoint_path):
        os.remove(checkpoint_path)

    print(f"\n  ✅ {split_name} annotations saved: {out_path}")
    print(f"     Total docs      : {len(completed)}")
    print(f"     Total sentences : {total_sents_processed:,}")
    _print_role_stats(completed)
    return completed


def _print_role_stats(annotations: list):
    counts = Counter()
    total  = 0
    for doc in annotations:
        for s in doc["sentences"]:
            counts[s["role"]] += 1
            total += 1
    print(f"\n  📊 Role Distribution ({total:,} sentences total):")
    print(f"  {'Role':<28} {'Count':>8} {'%':>7}")
    print("  " + "─" * 46)
    for role in LABELS_8:
        c   = counts[role]
        pct = c / total * 100 if total else 0
        bar = "█" * int(pct / 2)
        print(f"  {role:<28} {c:>8,} {pct:>6.1f}%  {bar}")

# ─────────────────────────────────────────────────────────
# STEP 2: ROLE CONTRIBUTION SCORES
# ─────────────────────────────────────────────────────────
def compute_role_contributions(train_annots: list, train_data: list,
                                out_path: str) -> dict:
    """
    RoleScore(r) = (1 / C_r) * Σ α_i
    Uses ALL training documents (no sample cap).
    """
    print(f"\n  Computing role contributions on {len(train_annots)} train docs...")
    total   = Counter()
    in_summ = Counter()

    for ann, item in tqdm(zip(train_annots, train_data),
                          total=len(train_annots), desc="  Contributions"):
        ref_sents = sent_tokenize(item.get("reference_summary", ""))
        doc_sents = [s["sentence"] for s in ann["sentences"]]
        doc_roles = [s["role"]     for s in ann["sentences"]]
        for r in doc_roles:
            total[r] += 1
        if doc_sents and ref_sents:
            doc_embs = embed_sentences(doc_sents)
            ref_embs = embed_sentences(ref_sents)
            for re in ref_embs:
                sims = cosine_similarity([re], doc_embs)[0]
                best = int(np.argmax(sims))
                if sims[best] > 0.7:
                    in_summ[doc_roles[best]] += 1

    scores = {r: in_summ[r] / total[r] if total[r] > 0 else 0.0
              for r in LABELS_8}
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump({
            "role_scores": scores,
            "role_total_counts":   dict(total),
            "role_summary_counts": dict(in_summ),
            "formula": "RoleScore(r) = (1/C_r) * Σ α_i",
            "train_docs_used": len(train_annots),
        }, f, indent=2, ensure_ascii=False)
    print(f"  ✅ Saved: {out_path}")
    _print_scores(scores, total, in_summ)
    return scores

def _print_scores(scores, total, in_summ):
    print(f"\n  {'Role':<28} {'Total':>8} {'InSumm':>8} {'Score':>8}")
    print("  " + "─" * 56)
    for r, sc in sorted(scores.items(), key=lambda x: -x[1]):
        print(f"  {r:<28} {total[r]:>8,} {in_summ[r]:>8,} {sc:>8.4f}")

# ─────────────────────────────────────────────────────────
# STEP 3: NORMALIZE ROLE WEIGHTS
# ─────────────────────────────────────────────────────────
def normalize_weights(scores: dict, out_path: str) -> dict:
    total = sum(scores.values()) or 1.0
    nw    = {r: s / total for r, s in scores.items()}
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump({"normalized_weights": nw, "original_scores": scores}, f,
                  indent=2, ensure_ascii=False)
    print(f"\n  ✅ Normalized weights saved: {out_path}")
    print(f"\n  {'Role':<28} {'Weight':>8}")
    print("  " + "─" * 38)
    for r, w in sorted(nw.items(), key=lambda x: -x[1]):
        print(f"  {r:<28} {w:>8.4f}")
    return nw

# ─────────────────────────────────────────────────────────
# HYBRID SELECTION  (LED)
# ─────────────────────────────────────────────────────────
def hybrid_selection(doc_ann: dict, norm_weights: dict, target: int):
    sents = [s["sentence"] for s in doc_ann["sentences"]]
    roles = [s["role"]     for s in doc_ann["sentences"]]
    if not sents:
        return "", {}
    embs = embed_sentences(sents)
    sal  = compute_saliency(sents, embs)
    scored = [
        {"idx": i,
         "score": HYBRID_ALPHA * norm_weights.get(r, 0) * 10 + HYBRID_BETA * float(sal[i]),
         "role": r}
        for i, r in enumerate(roles)
    ]
    selected = sorted(scored, key=lambda x: -x["score"])[:target]
    sel_idx  = sorted(s["idx"] for s in selected)
    return " ".join(sents[i] for i in sel_idx), {
        "method": "hybrid_cross_attention",
        "total": len(sents), "selected": len(sel_idx),
        "role_dist": dict(Counter(s["role"] for s in selected)),
    }

# ─────────────────────────────────────────────────────────
# RAG CONSTRUCTION  (BART, PEGASUS)
# ─────────────────────────────────────────────────────────
def rag_construct_input(doc_ann: dict, norm_weights: dict, model_name: str):
    sents = [s["sentence"] for s in doc_ann["sentences"]]
    roles = [s["role"]     for s in doc_ann["sentences"]]
    if not sents:
        return "", {}
    embs = embed_sentences(sents)
    sal  = compute_saliency(sents, embs)
    selected, used = [], set()

    # Stage 1 — Anchors
    s1 = {}
    for role in RAG_ANCHOR_ROLES:
        rs    = [(i, sents[i], roles[i]) for i in range(len(sents)) if roles[i] == role]
        n_anc = max(1, int(len(rs) * RAG_ANCHOR_FRACTION))
        for i, s, r in rs[-n_anc:]:
            if i not in used:
                selected.append((i, s, r, 1.0)); used.add(i)
        s1[role] = n_anc

    # Stage 2 — Role-quota (cross-attention saliency)
    total_w = sum(norm_weights.values()) or 1.0
    s2 = {}
    for role in LABELS_8:
        w      = norm_weights.get(role, 0.0)
        budget = max(1, int((w / total_w) * RAG_ROLE_BUDGET))
        cands  = [(i, sents[i], roles[i]) for i in range(len(sents))
                  if roles[i] == role and i not in used]
        top    = sorted(cands, key=lambda x: -float(sal[x[0]]))[:budget]
        for i, s, r in top:
            if i not in used:
                selected.append((i, s, r, float(sal[i]))); used.add(i)
        s2[role] = len(top)

    # Stage 3 — Narrative ordering
    ordered = sorted(selected, key=lambda x: (NARRATIVE_ORDER.get(x[2], 99), x[0]))
    return " ".join(x[1] for x in ordered), {
        "method": f"rag_cross_attention ({model_name})",
        "total": len(sents), "selected": len(ordered),
        "stage1": s1, "stage2": s2,
        "role_dist": dict(Counter(x[2] for x in ordered)),
    }

def build_input(doc_ann, norm_weights, model_name, adap):
    if model_name in RAG_MODELS:
        return rag_construct_input(doc_ann, norm_weights, model_name)
    return hybrid_selection(doc_ann, norm_weights, adap[model_name])

# ─────────────────────────────────────────────────────────
# SUMMARY GENERATION
# ─────────────────────────────────────────────────────────
def generate_summary(model_name: str, text: str) -> str:
    tok = summ_toks[model_name]
    cfg = MODEL_CONFIG[model_name]
    inp = tok(text, return_tensors="pt", truncation=True,
               max_length=cfg["max_input"]).to(device)
    kw  = dict(input_ids=inp["input_ids"],
               attention_mask=inp["attention_mask"],
               max_length=cfg["max_output"],
               num_beams=cfg["num_beams"],
               early_stopping=True, no_repeat_ngram_size=3)
    if model_name == "LED":
        gam = torch.zeros_like(inp["attention_mask"]); gam[:, 0] = 1
        kw["global_attention_mask"] = gam
    with torch.no_grad():
        ids = summ_models[model_name].generate(**kw)
    return tok.decode(ids[0], skip_special_tokens=True)

# ─────────────────────────────────────────────────────────
# ENSEMBLE STRATEGIES
# ─────────────────────────────────────────────────────────
def ensemble_voting(summs):
    all_s = [s for v in summs.values() for s in sent_tokenize(v)]
    cnt   = Counter(s.lower().strip() for s in all_s)
    sel   = [s for s, c in cnt.items() if c >= 2]
    return " ".join(sel) if sel else " ".join(s for s, _ in cnt.most_common(10))

def ensemble_weighted(summs, w):
    parts = []
    for m, v in summs.items():
        ss = sent_tokenize(v)
        parts.extend(ss[:max(1, int(len(ss) * w[m]))])
    seen, out = set(), []
    for s in parts:
        k = s.lower().strip()
        if k not in seen:
            seen.add(k); out.append(s)
    return " ".join(out)

def ensemble_ranking(summs):
    pm = {}
    for v in summs.values():
        for i, s in enumerate(sent_tokenize(v)):
            pm.setdefault(s.lower().strip(), []).append(i)
    return " ".join(s for s, _ in sorted(pm.items(), key=lambda x: np.mean(x[1]))[:15])

def ensemble_best(summs, ref):
    best_sc, best_s = -1.0, ""
    for v in summs.values():
        sc = rouge_metric.compute(predictions=[v], references=[ref],
                                   use_stemmer=True)["rouge2"]
        if sc > best_sc:
            best_sc, best_s = sc, v
    return best_s

# ─────────────────────────────────────────────────────────
# EVALUATION
# ─────────────────────────────────────────────────────────
def compute_metrics(preds, refs):
    r = rouge_metric.compute(predictions=preds, references=refs, use_stemmer=True)
    b = bertscore_metric.compute(predictions=preds, references=refs,
                                  model_type="roberta-large", lang="en",
                                  device=device, batch_size=8)
    return {"rouge1": float(r["rouge1"]), "rouge2": float(r["rouge2"]),
            "rougeL": float(r["rougeL"]), "bertscore_f1": float(np.mean(b["f1"]))}

def print_table(results):
    print(f"\n{'Approach':<28} {'R-1':>8} {'R-2':>8} {'R-L':>8} {'BERT-F1':>9}")
    print("─" * 60)
    for name, m in results.items():
        print(f"{name:<28} {m['rouge1']:>8.4f} {m['rouge2']:>8.4f}"
              f" {m['rougeL']:>8.4f} {m['bertscore_f1']:>9.4f}")

# ─────────────────────────────────────────────────────────
# MAIN PIPELINE
# ─────────────────────────────────────────────────────────
def main():
    print("\n" + "=" * 70)
    print("🏛️  HYBRID LEGAL SUMMARIZATION — FULL DATASET (7030 docs)")
    print("=" * 70)
    print(f"  ✅ No sample cap — ALL train documents used for annotation & scoring")
    print(f"  ✅ Checkpoint every {CHECKPOINT_EVERY} docs — safe to interrupt & resume")
    print(f"  Saliency : InLegalBERT + Multi-Head Cross-Sentence Self-Attention")
    print(f"  Scoring  : {HYBRID_ALPHA}×role_weight×10 + {HYBRID_BETA}×cross_attn_saliency")
    print("=" * 70)

    # ── Load data ────────────────────────────────────────────────
    print("\n📂 Loading data...")
    with open(TRAIN_JSON_PATH, encoding="utf-8") as f:
        train_data = json.load(f)
    if MAX_TRAIN_SAMPLES:
        train_data = train_data[:MAX_TRAIN_SAMPLES]

    with open(TEST_JSON_PATH, encoding="utf-8") as f:
        test_data = json.load(f)

    print(f"  Train docs : {len(train_data):,}")
    print(f"  Test docs  : {len(test_data):,}")
    print(f"  Total      : {len(train_data) + len(test_data):,}")

    # ── STEP 1: Annotate FULL train split ────────────────────────
    train_ann_path = os.path.join(OUTPUT_DIR, "train_" + ROLE_ANNOTATION_FILE)
    print(f"\n{'='*70}")
    print(f"STEP 1A: TRAIN ANNOTATION — {len(train_data):,} documents")
    print(f"{'='*70}")
    train_anns = annotate_documents_full(train_data, train_ann_path, "train")

    # ── STEP 1: Annotate FULL test split ─────────────────────────
    test_ann_path = os.path.join(OUTPUT_DIR, "test_" + ROLE_ANNOTATION_FILE)
    print(f"\n{'='*70}")
    print(f"STEP 1B: TEST ANNOTATION — {len(test_data):,} documents")
    print(f"{'='*70}")
    test_anns = annotate_documents_full(test_data, test_ann_path, "test")

    # ── STEP 2: Role contribution scores (full train) ────────────
    print(f"\n{'='*70}")
    print(f"STEP 2: ROLE CONTRIBUTION SCORES ({len(train_anns):,} train docs)")
    print(f"{'='*70}")
    contrib_path = os.path.join(OUTPUT_DIR, ROLE_CONTRIBUTION_FILE)
    if os.path.exists(contrib_path):
        print(f"  📂 Loading existing: {contrib_path}")
        with open(contrib_path, encoding="utf-8") as f:
            cd = json.load(f)
        role_scores = cd["role_scores"]
        _print_scores(role_scores, cd["role_total_counts"],
                      cd["role_summary_counts"])
    else:
        role_scores = compute_role_contributions(train_anns, train_data, contrib_path)

    # ── STEP 3: Normalize weights ────────────────────────────────
    print(f"\n{'='*70}")
    print("STEP 3: NORMALIZE ROLE WEIGHTS")
    print(f"{'='*70}")
    weights_path = os.path.join(OUTPUT_DIR, ROLE_WEIGHTS_FILE)
    if os.path.exists(weights_path):
        print(f"  📂 Loading existing: {weights_path}")
        with open(weights_path, encoding="utf-8") as f:
            norm_weights = json.load(f)["normalized_weights"]
    else:
        norm_weights = normalize_weights(role_scores, weights_path)

    # ── STEP 4: Generate summaries ───────────────────────────────
    print(f"\n{'='*70}")
    print(f"STEP 4: GENERATING SUMMARIES — {len(test_anns):,} test docs")
    print(f"{'='*70}")

    model_summaries = {"BART": [], "LED": [], "PEGASUS": []}
    ens_summaries   = {"voting": [], "weighted": [], "ranking": [], "best_model": []}
    references, adap_log, sel_samples = [], [], []

    for idx, (test_ann, test_item) in enumerate(
        tqdm(zip(test_anns, test_data), total=len(test_data),
             desc="  Generating", unit="doc")
    ):
        ref = test_item["reference_summary"]
        references.append(ref)
        adap = adaptive_targets(test_ann["total_sentences"])
        adap_log.append({"doc_id": test_ann["doc_id"],
                         "doc_len": test_ann["total_sentences"], "targets": adap})
        summs = {}
        for model_name in ("BART", "LED", "PEGASUS"):
            text, sel_info = build_input(test_ann, norm_weights, model_name, adap)
            if idx < 3:
                sel_samples.append({"doc_id": test_ann["doc_id"],
                                    "model": model_name, "info": sel_info})
            summary = generate_summary(model_name, text)
            model_summaries[model_name].append(summary)
            summs[model_name] = summary

        ens_summaries["voting"].append(ensemble_voting(summs))
        ens_summaries["weighted"].append(ensemble_weighted(summs, ENSEMBLE_WEIGHTS))
        ens_summaries["ranking"].append(ensemble_ranking(summs))
        ens_summaries["best_model"].append(ensemble_best(summs, ref))

    print("  ✅ All summaries generated.")

    with open(os.path.join(OUTPUT_DIR, "selection_samples.json"), "w") as f:
        json.dump(sel_samples, f, indent=2, ensure_ascii=False)

    # ── STEP 5: Evaluate ─────────────────────────────────────────
    print(f"\n{'='*70}")
    print("STEP 5: EVALUATING")
    print(f"{'='*70}")

    results = {}
    for m in ("BART", "LED", "PEGASUS"):
        print(f"  Evaluating {m}...")
        results[m] = compute_metrics(model_summaries[m], references)
        r = results[m]
        print(f"    R-1={r['rouge1']:.4f}  R-2={r['rouge2']:.4f}"
              f"  R-L={r['rougeL']:.4f}  BERT={r['bertscore_f1']:.4f}")

    for strat in ("voting", "weighted", "ranking", "best_model"):
        print(f"  Evaluating Ensemble-{strat}...")
        results[f"ensemble_{strat}"] = compute_metrics(
            ens_summaries[strat], references)

    print_table(results)
    best = max(results.items(), key=lambda x: x[1]["rouge2"])
    print(f"\n  🏆 Best : {best[0]}  "
          f"(R-1={best[1]['rouge1']:.4f} R-2={best[1]['rouge2']:.4f} "
          f"R-L={best[1]['rougeL']:.4f} BERT={best[1]['bertscore_f1']:.4f})")

    # ── STEP 6: Save ──────────────────────────────────────────────
    print(f"\n{'='*70}")
    print("STEP 6: SAVING OUTPUTS")
    print(f"{'='*70}")

    for m in ("BART", "LED", "PEGASUS"):
        meth = "rag_cross_attention" if m in RAG_MODELS else "hybrid_cross_attention"
        out  = os.path.join(OUTPUT_DIR, f"{m.lower()}_summaries.json")
        with open(out, "w", encoding="utf-8") as f:
            json.dump([
                {"id": item.get("id", i), "method": meth,
                 "generated_summary": summ, "reference_summary": ref,
                 "adaptive_target": adap_log[i]["targets"][m]}
                for i, (item, summ, ref) in enumerate(
                    zip(test_data, model_summaries[m], references))
            ], f, indent=2, ensure_ascii=False)
        print(f"  ✅ {os.path.basename(out)}")

    for strat in ("voting", "weighted", "ranking", "best_model"):
        out = os.path.join(OUTPUT_DIR, f"ensemble_{strat}_summaries.json")
        with open(out, "w", encoding="utf-8") as f:
            json.dump([
                {"id": item.get("id", i), "ensemble_method": strat,
                 "generated_summary": summ, "reference_summary": ref}
                for i, (item, summ, ref) in enumerate(
                    zip(test_data, ens_summaries[strat], references))
            ], f, indent=2, ensure_ascii=False)
        print(f"  ✅ {os.path.basename(out)}")

    with open(os.path.join(OUTPUT_DIR, "adaptive_targets_used.json"), "w") as f:
        json.dump(adap_log, f, indent=2, ensure_ascii=False)

    final = {
        "experiment": "Hybrid Legal Summarization V2 — Full Dataset",
        "dataset": {
            "train_docs": len(train_data),
            "test_docs":  len(test_data),
            "total":      len(train_data) + len(test_data),
            "sample_cap": "NONE — all documents used",
        },
        "saliency": "InLegalBERT + Multi-Head Cross-Sentence Self-Attention",
        "scoring":  f"{HYBRID_ALPHA}×role_weight×10 + {HYBRID_BETA}×cross_attn_saliency",
        "input_construction": {
            "BART":    "RAG 3-stage (anchor+quota+narrative)",
            "PEGASUS": "RAG 3-stage (anchor+quota+narrative)",
            "LED":     "Hybrid role-salience",
        },
        "ensemble_weights": ENSEMBLE_WEIGHTS,
        "results": results,
        "best_approach": {"name": best[0], "metrics": best[1]},
    }
    with open(os.path.join(OUTPUT_DIR, "complete_results.json"), "w") as f:
        json.dump(final, f, indent=2, ensure_ascii=False)
    print(f"  ✅ complete_results.json")

    print(f"\n✅ All outputs saved to: {OUTPUT_DIR}/")
    print("\n" + "=" * 70)
    print("✅ PIPELINE COMPLETE!")
    print("=" * 70)


if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\n⚠️  Interrupted. Checkpoint saved — re-run to resume annotation.")
    except Exception as e:
        import traceback
        print(f"\n❌ Error: {e}")
        traceback.print_exc()

🖥️  Device: cuda

📚 Loading models...
  [1/5] InLegalBERT...
  [2/5] Cross-Sentence Attention Scorer...
  [3/5] HSLN Role Classifier...
  [4/5] BART + LED + PEGASUS...
  [5/5] Evaluation metrics...
✅ All models loaded.


🏛️  HYBRID LEGAL SUMMARIZATION — FULL DATASET (7030 docs)
  ✅ No sample cap — ALL train documents used for annotation & scoring
  ✅ Checkpoint every 100 docs — safe to interrupt & resume
  Saliency : InLegalBERT + Multi-Head Cross-Sentence Self-Attention
  Scoring  : 0.6×role_weight×10 + 0.4×cross_attn_saliency

📂 Loading data...
  Train docs : 7,030
  Test docs  : 100
  Total      : 7,130

STEP 1A: TRAIN ANNOTATION — 7,030 documents

  📝 Annotating train: 7030 docs (total dataset = 7030)


  Annotating train:   1%|▍                              | 101/7030 [00:20<23:00,  5.02doc/s, sents_total=12435, roles=7]

  💾 Checkpoint saved: 100 / 7030 docs


  Annotating train:   3%|▉                              | 201/7030 [00:40<27:49,  4.09doc/s, sents_total=24933, roles=7]

  💾 Checkpoint saved: 200 / 7030 docs


...


  ✅ train annotations saved: ./results_cross_attention_ensemble/train_sentence_role_annotations_8label.json
     Total docs      : 7030
     Total sentences : 915,359

  📊 Role Distribution (915,359 sentences total):
  Role                            Count       %
  ──────────────────────────────────────────────
  PREAMBLE                       16,293    1.8%  
  FACTS                         153,067   16.7%  ████████
  ISSUE                          11,953    1.3%  
  ARGUMENTS                      23,903    2.6%  █
  REASONING                     404,958   44.2%  ██████████████████████
  PRECEDENT_RELIED               35,938    3.9%  █
  PRECEDENT_NOT_RELIED                7    0.0%  
  PROCEDURAL                    269,240   29.4%  ██████████████

STEP 1B: TEST ANNOTATION — 100 documents
  ✅ test annotation already complete (100 docs). Loading existing file.

STEP 2: ROLE CONTRIBUTION SCORES (7,030 train docs)
  📂 Loading existing: ./results_cross_attention_ensemble/role_contributi

  Generating: 100%|████████████████████████████████████████████████████████████████| 100/100 [1:04:30<00:00, 38.70s/doc]


  ✅ All summaries generated.

STEP 5: EVALUATING
  Evaluating BART...


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.weight', 'roberta.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


    R-1=0.3594  R-2=0.1803  R-L=0.2054  BERT=0.8491
  Evaluating LED...
    R-1=0.4999  R-2=0.2661  R-L=0.2808  BERT=0.8541
  Evaluating PEGASUS...
    R-1=0.3738  R-2=0.1840  R-L=0.2117  BERT=0.8440
  Evaluating Ensemble-voting...
  Evaluating Ensemble-weighted...
  Evaluating Ensemble-ranking...
  Evaluating Ensemble-best_model...

Approach                          R-1      R-2      R-L   BERT-F1
────────────────────────────────────────────────────────────
BART                           0.3594   0.1803   0.2054    0.8491
LED                            0.4999   0.2661   0.2808    0.8541
PEGASUS                        0.3738   0.1840   0.2117    0.8440
ensemble_voting                0.2646   0.1345   0.1455    0.8304
ensemble_weighted              0.4010   0.1990   0.2246    0.8446
ensemble_ranking               0.4703   0.2317   0.2338    0.8366
ensemble_best_model            0.4943   0.2745   0.2847    0.8568

  🏆 Best : ensemble_best_model  (R-1=0.4943 R-2=0.2745 R-L=0.2847 BERT=0.8